# 20 

In [1]:
import os, json, hashlib, importlib.util, urllib.request
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, confusion_matrix)

INPUT_ROOT = Path(os.environ.get("INPUT_ROOT", "/kaggle/input"))
DF_KEYS = Path(os.environ.get("DF_KEYS", INPUT_ROOT / "datasets/angelopaldino/df-keys/CM/trial_metadata.txt"))
NB19_SCORES = os.environ.get("NB19_SCORES", "")   # cartella 'punteggi_eval' del notebook 19; vuoto = ricerca automatica
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/metriche_pwfe")); OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELLI = {"PWfE": "PWfE_attuale", "PWfE_o": "PWfE_150_s0"}
RIF_EER = {"PWfE": 16.0872, "PWfE_o": 17.2025}    # dal notebook 19 (risultati_eval.csv)

def trova_punteggi():
    if NB19_SCORES:
        return Path(NB19_SCORES)
    hits = [Path(d) for d, _, f in os.walk(INPUT_ROOT)
            if Path(d).name == "punteggi_eval" and {"PWfE_attuale.npy", "PWfE_150_s0.npy"} <= set(f)]
    assert len(hits) == 1, f"cartelle 'punteggi_eval' trovate: {hits}; imposta NB19_SCORES"
    return hits[0]
SCO = trova_punteggi(); print("punteggi:", SCO)

EVAL_METRICS_URL = "https://raw.githubusercontent.com/asvspoof-challenge/2021/main/eval-package/eval_metrics.py"
p = OUT_DIR / "eval_metrics.py"
if not p.exists():
    urllib.request.urlretrieve(EVAL_METRICS_URL, p)
SHA = hashlib.sha256(p.read_bytes()).hexdigest(); assert SHA.startswith("70c5d2fe8fab6654"), SHA[:16]
spec = importlib.util.spec_from_file_location("asv_metrics", p); asv = importlib.util.module_from_spec(spec); spec.loader.exec_module(asv)

punteggi: /kaggle/input/datasets/angelo01paldino/pwfe-new/PWfE_riaddestrato_inferenza/pwfe_riaddestrato/punteggi_eval


In [2]:
COLS = ["speaker", "utt_id", "codec", "source", "attack", "label", "trim", "subset", "vocoder", "c9", "c10", "c11", "c12"]
meta = pd.read_csv(DF_KEYS, sep=r"\s+", header=None, names=COLS, dtype=str, engine="python")
meta = meta[meta["subset"] == "eval"].reset_index(drop=True)
Y = (meta["label"] == "bonafide").to_numpy().astype(int)
print(f"eval: {len(Y)} prove, {Y.sum()} bona fide, {(1 - Y).sum()} spoof")

def metriche(y, s, soglia):
    pred = (s > soglia).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    P, R, F, _ = precision_recall_fscore_support(y, pred, labels=[1, 0], zero_division=0)
    return {"soglia": float(soglia), "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
            "accuratezza_%": 100 * accuracy_score(y, pred),
            "accuratezza_bilanciata_%": 100 * balanced_accuracy_score(y, pred),
            "precision_bonafide_%": 100 * P[0], "recall_bonafide_%": 100 * R[0], "F1_bonafide_%": 100 * F[0],
            "precision_spoof_%": 100 * P[1], "recall_spoof_%": 100 * R[1], "F1_spoof_%": 100 * F[1],
            "F1_macro_%": 100 * F.mean(),
            "FPR_%": 100 * fp / (fp + tn), "FNR_%": 100 * fn / (fn + tp)}

righe = []
for nome, f in MODELLI.items():
    s = np.load(SCO / f"{f}.npy").astype(np.float64); assert len(s) == len(Y)
    eer, thr = asv.compute_eer(s[Y == 1], s[Y == 0])
    assert abs(100 * eer - RIF_EER[nome]) < 1e-3, f"{nome}: EER {100*eer:.4f} invece di {RIF_EER[nome]}"
    auc = roc_auc_score(Y, s)
    for tipo, t in (("soglia_EER", thr), ("logit_0", 0.0)):
        righe.append({"modello": nome, "file_punteggi": f, "tipo_soglia": tipo,
                      "EER_%": 100 * eer, "AUC": auc, **metriche(Y, s, t)})
RIS = pd.DataFrame(righe)
RIS.to_csv(OUT_DIR / "metriche_PWfE_PWfEo.csv", index=False)
print(f"riferimento banale 'sempre spoof': accuratezza {100 * (1 - Y.mean()):.2f}%")
pd.set_option("display.width", 250)
display(RIS.set_index(["modello", "tipo_soglia"]).drop(columns="file_punteggi").T.round(3))
(OUT_DIR / "info.json").write_text(json.dumps({"df_keys": str(DF_KEYS), "punteggi": str(SCO), "modelli": MODELLI,
    "eer_function": {"source": EVAL_METRICS_URL, "sha256": SHA},
    "convenzione": "bona fide = 1; bona fide se punteggio > soglia; FPR = spoof accettati; FNR = bona fide rifiutati"}, indent=2))
print("Output:", sorted(x.name for x in OUT_DIR.iterdir()))

eval: 533928 prove, 14869 bona fide, 519059 spoof
riferimento banale 'sempre spoof': accuratezza 97.22%


modello                         PWfE                  PWfE_o            
tipo_soglia               soglia_EER     logit_0  soglia_EER     logit_0
EER_%                         16.087      16.087      17.203      17.203
AUC                            0.914       0.914       0.902       0.902
soglia                        -7.551       0.000     -12.463       0.000
TP                         12477.000    8556.000   12311.000    8305.000
TN                        435557.000  514893.000  429773.000  515350.000
FP                         83502.000    4166.000   89286.000    3709.000
FN                          2392.000    6313.000    2558.000    6564.000
accuratezza_%                 83.913      98.037      82.798      98.076
accuratezza_bilanciata_%      83.913      78.370      82.797      77.570
precision_bonafide_%          13.000      67.254      12.117      69.128
recall_bonafide_%             83.913      57.543      82.796      55.854
F1_bonafide_%                 22.512      62.020      21.141      61.786
precision_spoof_%             99.454      98.789      99.408      98.742
recall_spoof_%                83.913      99.197      82.798      99.285
F1_spoof_%                    91.025      98.993      90.346      99.013
F1_macro_%                    56.768      80.506      55.744      80.400
FPR_%                         16.087       0.803      17.202       0.715
FNR_%                         16.087      42.457      17.204      44.146

Output: ['__pycache__', 'eval_metrics.py', 'info.json', 'metriche_PWfE_PWfEo.csv']
